# Day 11: Project 1 — Hospital Readmission Risk Predictor

**Business Problem:** US hospitals penalized for excess 30-day readmissions under HRRP. Model flags high-risk patients at discharge for targeted intervention.
**Dataset:** UCI Diabetes 130-US Hospitals (101,766 encounters)
**Target:** `readmitted` — "<30" vs ">30" + "NO" (binary)
**Goal:** Problem framing + data acquisition + first-pass EDA

In [1]:
import pandas as pd
import numpy as np
import os
import requests
import zipfile
import io

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# Download dataset from UCI
url = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
data_dir = "../data"
os.makedirs(data_dir, exist_ok=True)

zip_path = os.path.join(data_dir, "dataset_diabetes.zip")
csv_path = os.path.join(data_dir, "diabetes_raw.csv")

if not os.path.exists(csv_path):
    print("Downloading dataset...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(r.content)

    with zipfile.ZipFile(zip_path) as z:
        csv_member = next(
            name for name in z.namelist()
            if name.lower().endswith("diabetic_data.csv")
        )
        with z.open(csv_member) as source, open(csv_path, "wb") as target:
            target.write(source.read())

    print(f"Saved to {csv_path}")
else:
    print(f"Already exists: {csv_path}")

Saved to ../data\diabetes_raw.csv


In [2]:
# Load and inspect
df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}: {df[c].dtype}")

print("\n=== First 5 rows ===")
print(df.head())

Shape: (101766, 50)

Columns (50):
  encounter_id: int64
  patient_nbr: int64
  race: object
  gender: object
  age: object
  weight: object
  admission_type_id: int64
  discharge_disposition_id: int64
  admission_source_id: int64
  time_in_hospital: int64
  payer_code: object
  medical_specialty: object
  num_lab_procedures: int64
  num_procedures: int64
  num_medications: int64
  number_outpatient: int64
  number_emergency: int64
  number_inpatient: int64
  diag_1: object
  diag_2: object
  diag_3: object
  number_diagnoses: int64
  max_glu_serum: object
  A1Cresult: object
  metformin: object
  repaglinide: object
  nateglinide: object
  chlorpropamide: object
  glimepiride: object
  acetohexamide: object
  glipizide: object
  glyburide: object
  tolbutamide: object
  pioglitazone: object
  rosiglitazone: object
  acarbose: object
  miglitol: object
  troglitazone: object
  tolazamide: object
  examide: object
  citoglipton: object
  insulin: object
  glyburide-metformin: object
  g

In [3]:
# Target variable exploration
print("=== readmitted value counts ===")
print(df['readmitted'].value_counts())
print(df['readmitted'].value_counts(normalize=True).round(4))

=== readmitted value counts ===
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
readmitted
NO     0.5391
>30    0.3493
<30    0.1116
Name: proportion, dtype: float64


In [4]:
# Binary target: readmitted <30 days = 1, else 0
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

print("=== Binary target distribution ===")
print(df['readmitted_binary'].value_counts())
print(df['readmitted_binary'].value_counts(normalize=True).round(4))

=== Binary target distribution ===
readmitted_binary
0    90409
1    11357
Name: count, dtype: int64
readmitted_binary
0    0.8884
1    0.1116
Name: proportion, dtype: float64


In [13]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

print(missing_pct)

missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print("=== Missing Values ===")
print(missing_df)

encounter_id                 0.00
patient_nbr                  0.00
race                         0.00
gender                       0.00
age                          0.00
weight                       0.00
admission_type_id            0.00
discharge_disposition_id     0.00
admission_source_id          0.00
time_in_hospital             0.00
payer_code                   0.00
medical_specialty            0.00
num_lab_procedures           0.00
num_procedures               0.00
num_medications              0.00
number_outpatient            0.00
number_emergency             0.00
number_inpatient             0.00
diag_1                       0.00
diag_2                       0.00
diag_3                       0.00
number_diagnoses             0.00
max_glu_serum               94.75
A1Cresult                   83.28
metformin                    0.00
repaglinide                  0.00
nateglinide                  0.00
chlorpropamide               0.00
glimepiride                  0.00
acetohexamide 

In [14]:
# Categorical column exploration
cat_cols = df.select_dtypes(include='object').columns
print(f"Categorical columns ({len(cat_cols)}):")
for col in cat_cols:
    n_unique = df[col].nunique()
    print(f"  {col}: {n_unique} unique")
    if n_unique <= 20:
        print(f"    Values: {df[col].unique()}")

Categorical columns (37):
  race: 6 unique
    Values: ['Caucasian' 'AfricanAmerican' '?' 'Other' 'Asian' 'Hispanic']
  gender: 3 unique
    Values: ['Female' 'Male' 'Unknown/Invalid']
  age: 10 unique
    Values: ['[0-10)' '[10-20)' '[20-30)' '[30-40)' '[40-50)' '[50-60)' '[60-70)'
 '[70-80)' '[80-90)' '[90-100)']
  weight: 10 unique
    Values: ['?' '[75-100)' '[50-75)' '[0-25)' '[100-125)' '[25-50)' '[125-150)'
 '[175-200)' '[150-175)' '>200']
  payer_code: 18 unique
    Values: ['?' 'MC' 'MD' 'HM' 'UN' 'BC' 'SP' 'CP' 'SI' 'DM' 'CM' 'CH' 'PO' 'WC' 'OT'
 'OG' 'MP' 'FR']
  medical_specialty: 73 unique
  diag_1: 717 unique
  diag_2: 749 unique
  diag_3: 790 unique
  max_glu_serum: 3 unique
    Values: [nan '>300' 'Norm' '>200']
  A1Cresult: 3 unique
    Values: [nan '>7' '>8' 'Norm']
  metformin: 4 unique
    Values: ['No' 'Steady' 'Up' 'Down']
  repaglinide: 4 unique
    Values: ['No' 'Up' 'Steady' 'Down']
  nateglinide: 4 unique
    Values: ['No' 'Steady' 'Down' 'Up']
  chlorpropamid

In [15]:
# Numeric summary
num_cols = df.select_dtypes(include=[np.number]).columns
print(df[num_cols].describe().T)

                             count          mean           std      min         25%          50%           75%          max
encounter_id              101766.0  1.652016e+08  1.026403e+08  12522.0  84961194.0  152388987.0  2.302709e+08  443867222.0
patient_nbr               101766.0  5.433040e+07  3.869636e+07    135.0  23413221.0   45505143.0  8.754595e+07  189502619.0
admission_type_id         101766.0  2.024006e+00  1.445403e+00      1.0         1.0          1.0  3.000000e+00          8.0
discharge_disposition_id  101766.0  3.715642e+00  5.280166e+00      1.0         1.0          1.0  4.000000e+00         28.0
admission_source_id       101766.0  5.754437e+00  4.064081e+00      1.0         1.0          7.0  7.000000e+00         25.0
time_in_hospital          101766.0  4.395987e+00  2.985108e+00      1.0         2.0          4.0  6.000000e+00         14.0
num_lab_procedures        101766.0  4.309564e+01  1.967436e+01      1.0        31.0         44.0  5.700000e+01        132.0
num_proc